# LeNet-5 Neural Network for MNIST

### Options

In [1]:
batch_size = 20
training_epochs = 10

### Load & convert data

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

def convert_data(x_data, y_data):
    # hacky way to increase the size of a (unknown)x28x28 to (unknown)x32x32
    # by appending two zeros at all sides of the 28x28 matrix
    tmp = np.zeros((x_data.shape[0], 2, x_data.shape[2]))
    x_data = np.append(x_data, tmp, axis=1)
    x_data = np.append(tmp, x_data, axis=1)
    tmp = np.zeros((x_data.shape[0], x_data.shape[1], 2))
    x_data = np.append(x_data, tmp, axis=2)
    x_data = np.append(tmp, x_data, axis=2)

    # change dimensions: (unknown)x32x32 to (unknown)x32x32x1
    x_data = np.expand_dims(x_data, -1)
    # change dimensions: (unknown) to (unknown)x10
    y_data = keras.utils.to_categorical(y_data, 10)

    return (x_data, y_data)

# for now we are just interested in the training data and not the test data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

(X_train, y_train) = convert_data(X_train, y_train)
(X_test, y_test) = convert_data(X_test, y_test)

2024-10-23 21:39:54.692534: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-10-23 21:39:54.698422: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-10-23 21:39:54.718378: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-23 21:39:54.751914: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-23 21:39:54.761775: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-23 21:39:54.786669: I tensorflow/core/platform/cpu_feature_gu

In [3]:
print("Shapes:")
print((X_train.shape, y_train.shape))

Shapes:
((60000, 32, 32, 1), (60000, 10))


### Setup our model

In [4]:
model = keras.Sequential([
    keras.layers.Input(shape=(32, 32, 1)),
    keras.layers.Conv2D(6, kernel_size=(5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(16, kernel_size=(5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(120),
    keras.layers.Dense(84),
    keras.layers.Dense(10),
    keras.layers.Activation('softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

### Fitting the model

In [5]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

hist = model.fit(X_train, y_train, batch_size=batch_size, epochs=training_epochs, validation_split=0.2)

Epoch 1/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - accuracy: 0.8573 - loss: 3.0759 - val_accuracy: 0.9565 - val_loss: 0.1506
Epoch 2/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.9607 - loss: 0.1308 - val_accuracy: 0.9665 - val_loss: 0.1113
Epoch 3/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9668 - loss: 0.1151 - val_accuracy: 0.9735 - val_loss: 0.0890
Epoch 4/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - accuracy: 0.9675 - loss: 0.1138 - val_accuracy: 0.9725 - val_loss: 0.0875
Epoch 5/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - accuracy: 0.9722 - loss: 0.0911 - val_accuracy: 0.9755 - val_loss: 0.0816
Epoch 6/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - accuracy: 0.9756 - loss: 0.0785 - val_accuracy: 0.9749 - val_loss: 0.0937
Epoch 7/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.9781 - loss: 0.0726 - val_accuracy: 0.9735 - val_loss: 0.0891
Epoch 8/10
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.9817 - loss: 0

## How good is our model for non-train data?

In [8]:
score = model.evaluate(X_test, y_test, batch_size=batch_size)

score

500/500 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9704 - loss: 0.0983


[0.08761253952980042, 0.973800003528595]